In [2]:
#import sys
#!{sys.executable} -m pip install "psycopg[binary]" pgvector

In [103]:
# Standard libraries
import os
import re
import json
import hashlib
import warnings
from pathlib import Path

# Data processing
import numpy as np
import pandas as pd

# PyTorch
import torch

# Progress bar
from tqdm.auto import tqdm

# LangChain documents and text splitters
from langchain_core.documents import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# PostgreSQL and pgvector
import psycopg
from pgvector.psycopg import register_vector

# Notebook settings
warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.8.0+cu129
Device: cpu


## Connect to PostgreSQL

In [110]:
# Connect to PostgreSQL
connection = psycopg.connect(dbname="postgres", user="postgres", password="root", host="localhost", port="5432")
cursor = connection.cursor()
register_vector(connection)
print("Connected to PostgreSQL.")

Connected to PostgreSQL.


## Load Markdown file / files

In [120]:
# Generate file hash
def get_file_hash(file_path):
    content = file_path.read_bytes()
    file_hash = hashlib.sha256(content).hexdigest()

    return file_hash

In [121]:
# Load Markdown files that require processing
def load_markdown_files(data_path=Path("data")):
    markdown_files = []

    for file_path in data_path.glob("*.md"):
        file_hash = get_file_hash(file_path)

        cursor.execute("SELECT file_hash FROM documents WHERE filename = %s;", (file_path.name,))

        result = cursor.fetchone()

        if result is None:
            print("New:", file_path.name)
            markdown_files.append(file_path)

        elif result[0] != file_hash:
            print("Modified:", file_path.name)
            markdown_files.append(file_path)

        else:
            print("Skipping unchanged:", file_path.name)

    return markdown_files

In [122]:
markdown_files = load_markdown_files()
print("Files to process:", len(markdown_files))

New: adblue-and-def.md
New: americas-biofuels.md
Files to process: 2


In [123]:
for file_path in markdown_files:
    markdown_text = file_path.read_text(encoding="utf-8", errors="replace")

    print("File:", file_path.name)
    print("Characters:", len(markdown_text))
    print("Words:", len(markdown_text.split()))
    print("Lines:", len(markdown_text.splitlines()))
    print()

File: adblue-and-def.md
Characters: 23443
Words: 3479
Lines: 269

File: americas-biofuels.md
Characters: 69791
Words: 10723
Lines: 1060



## Split the document by Markdown headings

In [124]:
headers_to_split_on = [
    ("#", "document_title"),
    ("##", "section"),
    ("###", "subsection"),
    ("####", "subsubsection"),
    ("#####", "detail"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)

sections = []

for file_path in markdown_files:
    markdown_text = file_path.read_text(encoding="utf-8", errors="replace")

    file_sections = markdown_splitter.split_text(markdown_text)

    for section in file_sections:
        section.metadata["source"] = file_path.name
        sections.append(section)

    print("File:", file_path.name)
    print("Sections:", len(file_sections))
    print()

print("Total sections:", len(sections))

File: adblue-and-def.md
Sections: 30

File: americas-biofuels.md
Sections: 118

Total sections: 148


In [127]:
# Inspect five sections from each file
for file_path in markdown_files:
    count = 0

    print("\nFILE:", file_path.name)

    for section in sections:
        if section.metadata["source"] == file_path.name:
            print("SECTION:", count + 1)
            print("Metadata:", section.metadata)
            print("Characters:", len(section.page_content))
            print("Content:", section.page_content[:200])
            print()

            count += 1

            if count == 5:
                break


FILE: adblue-and-def.md
SECTION: 1
Metadata: {'document_title': 'METHODOLOGY AND SPECIFICATIONS GUIDE', 'section': 'ARGUS ADBLUE®-DEF AND TGU', 'subsection': 'Contents:', 'source': 'adblue-and-def.md'}
Characters: 346
Content: # METHODOLOGY AND SPECIFICATIONS GUIDE  
## ARGUS ADBLUE®-DEF AND TGU  
### Contents:  
*   Methodology overview
*   Argus AdBlue®-DEF and TGU
*   Product specification
*   Spot pricing  
---  
**LAST

SECTION: 2
Metadata: {'document_title': 'METHODOLOGY AND SPECIFICATIONS GUIDE', 'section': 'Methodology overview', 'subsection': 'Methodology rationale', 'source': 'adblue-and-def.md'}
Characters: 1057
Content: ## Methodology overview  
### Methodology rationale  
Argus strives to construct methodologies that reflect the way the market trades. Argus aims to produce price assessments which are reliable and re

SECTION: 3
Metadata: {'document_title': 'METHODOLOGY AND SPECIFICATIONS GUIDE', 'section': 'Methodology overview', 'subsection': 'Survey process', 'source': 

## Add heading path metadata

In [128]:
# Add heading path metadata
for section in sections:

    heading_parts = [
        section.metadata.get("document_title"),
        section.metadata.get("section"),
        section.metadata.get("subsection"),
        section.metadata.get("subsubsection"),
        section.metadata.get("detail"),
    ]

    heading_path = []

    for part in heading_parts:
        if part:
            heading_path.append(part)

    section.metadata["heading_path"] = " > ".join(heading_path)

In [130]:
# Inspect three metadata entries from each file
for file_path in markdown_files:
    count = 0

    print("File:", file_path.name)

    for section in sections:
        if section.metadata["source"] == file_path.name:
            print(section.metadata)
            print()

            count += 1

            if count == 3:
                break

File: adblue-and-def.md
{'document_title': 'METHODOLOGY AND SPECIFICATIONS GUIDE', 'section': 'ARGUS ADBLUE®-DEF AND TGU', 'subsection': 'Contents:', 'source': 'adblue-and-def.md', 'heading_path': 'METHODOLOGY AND SPECIFICATIONS GUIDE > ARGUS ADBLUE®-DEF AND TGU > Contents:'}

{'document_title': 'METHODOLOGY AND SPECIFICATIONS GUIDE', 'section': 'Methodology overview', 'subsection': 'Methodology rationale', 'source': 'adblue-and-def.md', 'heading_path': 'METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Methodology rationale'}

{'document_title': 'METHODOLOGY AND SPECIFICATIONS GUIDE', 'section': 'Methodology overview', 'subsection': 'Survey process', 'source': 'adblue-and-def.md', 'heading_path': 'METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Survey process'}

File: americas-biofuels.md
{'document_title': 'ARGUS AMERICAS BIOFUELS', 'section': 'Contents:', 'source': 'americas-biofuels.md', 'heading_path': 'ARGUS AMERICAS BIOFUELS > Contents:'}

{'document_ti

## Inspect section sizes

In [131]:
# Inspect section sizes
section_sizes = []
for index, section in enumerate(sections, start=1):
    char_count = len(section.page_content)
    word_count = len(section.page_content.split())
    section_sizes.append({
        "section": index,
        "source": section.metadata["source"],
        "chars": char_count,
        "words": word_count,
        "heading_path": section.metadata["heading_path"]
    })

In [132]:
section_sizes_df = pd.DataFrame(section_sizes)
section_sizes_df.sort_values("chars", ascending=False).head(5)

,section,source,chars,words,heading_path
33,34,americas-biofuels.md,3403,527,ARGUS AMERICAS BIOFUELS > Methodology overview > Survey process
44,45,americas-biofuels.md,2007,291,ARGUS AMERICAS BIOFUELS > Methodology overview > Volume minimums and transaction data thresholds
31,32,americas-biofuels.md,1906,291,ARGUS AMERICAS BIOFUELS > Methodology overview > Methodology rationale
54,55,americas-biofuels.md,1822,268,ARGUS AMERICAS BIOFUELS > Argus publishes various price types for each commodity. These typically include > Changes to methodology
20,21,adblue-and-def.md,1754,258,METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Changes to methodology


## RecursiveCharacterTextSplitter

In [134]:
# Split only large sections
final_sections = []
split_files = []

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150
)

for section in sections:
    if len(section.page_content) > 1800:

        if section.metadata["source"] not in split_files:
            split_files.append(section.metadata["source"])

        split_sections = recursive_splitter.split_documents([section])

        for split_section in split_sections:
            final_sections.append(split_section)

    else:
        final_sections.append(section)

print("Original sections:", len(sections))
print("Final sections:", len(final_sections))
print("Files recursively split:", split_files)

Original sections: 148
Final sections: 153
Files recursively split: ['americas-biofuels.md']


## Add chunk IDs

In [135]:
# Add chunk IDs
for index, section in enumerate(final_sections):
    section.metadata["chunk_id"] = index

In [136]:
print(final_sections[5].metadata)
print()
print(final_sections[5].page_content)

{'document_title': 'METHODOLOGY AND SPECIFICATIONS GUIDE', 'section': 'Methodology overview', 'subsection': 'Verification of transaction data', 'subsubsection': 'Primary tests applied by reporters', 'source': 'adblue-and-def.md', 'heading_path': 'METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Verification of transaction data > Primary tests applied by reporters', 'chunk_id': 5}

#### Primary tests applied by reporters  
*   Transactions not transacted at arm's length, including deals between related parties or affiliates.
*   Transaction prices that deviate significantly from the mean of all transactions submitted for that day.
*   Transaction prices that fall outside of the generally observed lows and highs that operated throughout the trading day.
*   Transactions that are suspected to be a leg of another transaction or in some way contingent on an unknown transaction.
*   Single deal volumes that significantly exceed the typical transaction volume for that market.
*  

In [137]:
for section in final_sections:
    if section.metadata["heading_path"].endswith("Survey process"):
        print(section.metadata)
        print(len(section.page_content))
        print()

{'document_title': 'METHODOLOGY AND SPECIFICATIONS GUIDE', 'section': 'Methodology overview', 'subsection': 'Survey process', 'source': 'adblue-and-def.md', 'heading_path': 'METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Survey process', 'chunk_id': 2}
1586

{'document_title': 'ARGUS AMERICAS BIOFUELS', 'section': 'Methodology overview', 'subsection': 'Survey process', 'source': 'americas-biofuels.md', 'heading_path': 'ARGUS AMERICAS BIOFUELS > Methodology overview > Survey process', 'chunk_id': 34}
1396

{'document_title': 'ARGUS AMERICAS BIOFUELS', 'section': 'Methodology overview', 'subsection': 'Survey process', 'source': 'americas-biofuels.md', 'heading_path': 'ARGUS AMERICAS BIOFUELS > Methodology overview > Survey process', 'chunk_id': 35}
1263

{'document_title': 'ARGUS AMERICAS BIOFUELS', 'section': 'Methodology overview', 'subsection': 'Survey process', 'source': 'americas-biofuels.md', 'heading_path': 'ARGUS AMERICAS BIOFUELS > Methodology overview > Survey pr

## Add contextual information to each chunk

In [138]:
# Add contextual information to each chunk
contextual_chunks = []

for section in final_sections:
    source = section.metadata["source"]
    heading_path = section.metadata["heading_path"]

    contextual_text = "Source: " + source + "\n"
    contextual_text += "Context: " + heading_path + "\n\n"
    contextual_text += section.page_content

    contextual_chunk = Document(page_content=contextual_text, metadata=section.metadata.copy())

    contextual_chunks.append(contextual_chunk)

In [139]:
# Compare original and contextualized chunk
print("ORIGINAL:\n")
print(final_sections[5].page_content)

print("============================================================")

print("\nCONTEXTUALIZED:\n")
print(contextual_chunks[5].page_content)

ORIGINAL:

#### Primary tests applied by reporters  
*   Transactions not transacted at arm's length, including deals between related parties or affiliates.
*   Transaction prices that deviate significantly from the mean of all transactions submitted for that day.
*   Transaction prices that fall outside of the generally observed lows and highs that operated throughout the trading day.
*   Transactions that are suspected to be a leg of another transaction or in some way contingent on an unknown transaction.
*   Single deal volumes that significantly exceed the typical transaction volume for that market.
*   Transaction details that are identified by other market participants as being for any reason potentially anomalous and perceived by Argus to be as such.
*   Transaction details that are reported by one counterparty differently than the other counterparty.
*   Any transaction details that appear to the reporter to be illogical or to stray from the norms of trading behaviour. This cou

## Initialize embedding model

In [140]:
# Initialize embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 8186.70it/s]


In [141]:
# Generate one test embedding
test_embedding = embedding_model.embed_query(contextual_chunks[5].page_content)
print("Embedding dimensions:", len(test_embedding))
print(test_embedding[:10])

Embedding dimensions: 768
[-0.0038097051437944174, -0.011952378787100315, -0.021331004798412323, 0.013023106381297112, 0.07678933441638947, 0.0016202188562601805, 0.010243838652968407, 0.02343219518661499, 0.02451828308403492, -0.030195539817214012]


In [142]:
print(contextual_chunks[5].metadata["chunk_id"])
print(contextual_chunks[5].metadata["heading_path"])

5
METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Verification of transaction data > Primary tests applied by reporters


In [143]:
# Ingest file
def ingest_file(file_path, chunks):
    file_hash = get_file_hash(file_path)

    cursor.execute("SELECT id FROM documents WHERE filename = %s;", (file_path.name,))

    result = cursor.fetchone()

    if result is not None:
        cursor.execute("DELETE FROM documents WHERE id = %s;", (result[0],))

    cursor.execute("INSERT INTO documents (filename, file_hash) VALUES (%s, %s) RETURNING id;", (file_path.name, file_hash))

    document_id = cursor.fetchone()[0]

    texts = []

    for chunk in chunks:
        texts.append(chunk.page_content)

    embeddings = embedding_model.embed_documents(texts)

    for index, chunk in enumerate(chunks):
        cursor.execute(
            """
            INSERT INTO chunks (
                document_id,
                chunk_id,
                heading_path,
                content,
                embedding
            )
            VALUES (%s, %s, %s, %s, %s);
            """,
            (
                document_id,
                index,
                chunk.metadata["heading_path"],
                chunk.page_content,
                embeddings[index]
            )
        )

    connection.commit()

    print("Indexed:", file_path.name)

In [144]:
# Ingest files
for file_path in markdown_files:
    file_chunks = []

    for chunk in contextual_chunks:
        if chunk.metadata["source"] == file_path.name:
            file_chunks.append(chunk)

    ingest_file(file_path, file_chunks)

Indexed: adblue-and-def.md
Indexed: americas-biofuels.md
